# 5.2 Backtracking search for CSPs: practice notebook

Solve CSPs by assigning one variable at a time and going back as soon as a rule breaks. Choose the next variable with MRV and degree, the value with LCV, and prune ahead with forward checking and MAC.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Plain backtracking on the map of Australia

In [ ]:
# Plain backtracking on the map of Australia, fixed order and fixed colour order.
order = ["WA", "NT", "NSW", "Q", "SA", "V", "T"]
neighbours = {"WA": ["NT", "SA"], "NT": ["WA", "SA", "Q"], "SA": ["WA", "NT", "Q", "NSW", "V"],
              "Q": ["NT", "SA", "NSW"], "NSW": ["Q", "SA", "V"], "V": ["SA", "NSW"], "T": []}
colours = ["red", "green", "blue"]
tries = 0

def backtrack(assignment):
    global tries
    if len(assignment) == len(order):
        return assignment
    var = order[len(assignment)]               # next variable in the fixed order
    for c in colours:
        tries = tries + 1
        clash = False
        for n in neighbours[var]:
            if assignment.get(n) == c:
                clash = True
        if not clash:
            assignment[var] = c
            result = backtrack(assignment)
            if result is not None:
                return result
            del assignment[var]                # undo and try the next colour
            print("backtrack: undo", var, "=", c)
    return None

print(backtrack({}))
print("colours tried:", tries)

### Backtracking with MRV and forward checking

In [ ]:
# Backtracking with MRV (fewest values left) and forward checking on the same map.
neighbours = {"WA": ["NT", "SA"], "NT": ["WA", "SA", "Q"], "SA": ["WA", "NT", "Q", "NSW", "V"],
              "Q": ["NT", "SA", "NSW"], "NSW": ["Q", "SA", "V"], "V": ["SA", "NSW"], "T": []}

def solve(assignment, domains):
    if len(assignment) == len(neighbours):
        return assignment
    # MRV: the unassigned variable with the fewest values left (ties: most neighbours)
    best = None
    for v in neighbours:
        if v not in assignment:
            if best is None or len(domains[v]) < len(domains[best]) or \
               (len(domains[v]) == len(domains[best]) and len(neighbours[v]) > len(neighbours[best])):
                best = v
    for c in domains[best]:
        new_domains = {}
        for v in domains:
            new_domains[v] = list(domains[v])
        new_domains[best] = [c]
        dead = False
        for n in neighbours[best]:              # forward checking
            if n not in assignment and c in new_domains[n]:
                new_domains[n].remove(c)
                if len(new_domains[n]) == 0:
                    dead = True
        print("choose", best, "=", c, " dead end" if dead else "")
        if not dead:
            assignment[best] = c
            result = solve(assignment, new_domains)
            if result is not None:
                return result
            del assignment[best]
    return None

start = {}
for v in neighbours:
    start[v] = ["red", "green", "blue"]
print(solve({}, start))

## Your turn

### Task 1: Backtracking map colouring

Write `colour_map(order, neighbours, colours)` that colours the variables in the given order, trying colours in the given order, and backtracks when stuck. Return the assignment as a dict, or None if there is none.

Hint: Use a helper backtrack(i) that colours order[i] and calls backtrack(i + 1). Delete the colour when you undo.

In [ ]:
def colour_map(order, neighbours, colours):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("colour_map(['WA', 'NT', 'NSW', 'Q', 'SA', 'V', 'T'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []}, ['red', 'green', 'blue'])", {'WA': 'red', 'NT': 'green', 'NSW': 'green', 'Q': 'red', 'SA': 'blue', 'V': 'red', 'T': 'red'}), ("colour_map(['WA', 'NT', 'NSW', 'Q', 'SA', 'V', 'T'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []}, ['red', 'green'])", None), ("colour_map(['A', 'B'], {'A': ['B'], 'B': ['A']}, ['x', 'y'])", {'A': 'x', 'B': 'y'})])

### Task 2: Pick the next variable with MRV and degree

Write `pick(domains, assigned, neighbours)`. Among unassigned variables, return the one with the fewest values in domains. Break ties by the most unassigned neighbours, then by the order the variables appear in domains.

Hint: Keep the best so far, and replace it only if the new one has fewer values, or the same number and a higher degree.

In [ ]:
def pick(domains, assigned, neighbours):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("pick({'WA': ['r', 'g', 'b'], 'NT': ['r', 'g', 'b'], 'SA': ['r', 'g', 'b'], 'Q': ['r', 'g', 'b'], 'NSW': ['r', 'g', 'b'], 'V': ['r', 'g', 'b'], 'T': ['r', 'g', 'b']}, [], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []})", 'SA'), ("pick({'WA': ['r'], 'NT': ['g'], 'SA': ['b'], 'Q': ['r', 'b'], 'NSW': ['r', 'g', 'b'], 'V': ['r', 'g', 'b'], 'T': ['r', 'g', 'b']}, ['WA', 'NT'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []})", 'SA'), ("pick({'WA': ['r'], 'NT': ['g'], 'SA': ['b'], 'Q': ['r'], 'NSW': ['g', 'b'], 'V': ['r', 'g'], 'T': ['r', 'g', 'b']}, ['WA', 'NT', 'SA', 'Q'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []})", 'NSW')])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Backtracking map colouring

The answer matches the animation: two backtracks, then NSW = green.

In [ ]:
def colour_map(order, neighbours, colours):
    assignment = {}

    def backtrack(i):
        if i == len(order):
            return True
        var = order[i]
        for c in colours:
            clash = False
            for n in neighbours[var]:
                if assignment.get(n) == c:
                    clash = True
            if not clash:
                assignment[var] = c
                if backtrack(i + 1):
                    return True
                del assignment[var]
        return False

    if backtrack(0):
        return assignment
    return None

In [ ]:
check([("colour_map(['WA', 'NT', 'NSW', 'Q', 'SA', 'V', 'T'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []}, ['red', 'green', 'blue'])", {'WA': 'red', 'NT': 'green', 'NSW': 'green', 'Q': 'red', 'SA': 'blue', 'V': 'red', 'T': 'red'}), ("colour_map(['WA', 'NT', 'NSW', 'Q', 'SA', 'V', 'T'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []}, ['red', 'green'])", None), ("colour_map(['A', 'B'], {'A': ['B'], 'B': ['A']}, ['x', 'y'])", {'A': 'x', 'B': 'y'})])

### Solution to task 2: Pick the next variable with MRV and degree

At the start all domains are equal, so degree decides: SA. In the last test NSW and V both have 2 values, and NSW has the higher degree (V is its only unassigned neighbour, and T has 3 values).

In [ ]:
def pick(domains, assigned, neighbours):
    best = None
    best_size = None
    best_degree = None
    for v in domains:
        if v in assigned:
            continue
        size = len(domains[v])
        degree = 0
        for n in neighbours[v]:
            if n not in assigned:
                degree = degree + 1
        if best is None or size < best_size or (size == best_size and degree > best_degree):
            best = v
            best_size = size
            best_degree = degree
    return best

In [ ]:
check([("pick({'WA': ['r', 'g', 'b'], 'NT': ['r', 'g', 'b'], 'SA': ['r', 'g', 'b'], 'Q': ['r', 'g', 'b'], 'NSW': ['r', 'g', 'b'], 'V': ['r', 'g', 'b'], 'T': ['r', 'g', 'b']}, [], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []})", 'SA'), ("pick({'WA': ['r'], 'NT': ['g'], 'SA': ['b'], 'Q': ['r', 'b'], 'NSW': ['r', 'g', 'b'], 'V': ['r', 'g', 'b'], 'T': ['r', 'g', 'b']}, ['WA', 'NT'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []})", 'SA'), ("pick({'WA': ['r'], 'NT': ['g'], 'SA': ['b'], 'Q': ['r'], 'NSW': ['g', 'b'], 'V': ['r', 'g'], 'T': ['r', 'g', 'b']}, ['WA', 'NT', 'SA', 'Q'], {'WA': ['NT', 'SA'], 'NT': ['WA', 'SA', 'Q'], 'SA': ['WA', 'NT', 'Q', 'NSW', 'V'], 'Q': ['NT', 'SA', 'NSW'], 'NSW': ['Q', 'SA', 'V'], 'V': ['SA', 'NSW'], 'T': []})", 'NSW')])